In [5]:
!nvidia-smi
!python --version
!pip install datasets tokenizers numpy

Sat Mar  7 19:14:12 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 581.83                 Driver Version: 581.83         CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce GTX 1650      WDDM  |   00000000:01:00.0 Off |                  N/A |
| N/A   43C    P8              4W /   30W |       0MiB /   4096MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [6]:
!pip install datasets huggingface_hub


In [7]:
from datasets import load_dataset

print("Loading medalpaca...")
medalpaca = load_dataset("medalpaca/medical_meadow_medqa", split="train")

print("Loading pubmed_qa...")
pubmed = load_dataset("pubmed_qa", "pqa_labeled", split="train", trust_remote_code=True)

print("Loading medmcqa...")
medmcqa = load_dataset("medmcqa", split="train")

print("✅ All datasets loaded!")
print(f"  medalpaca: {len(medalpaca)} rows")
print(f"  pubmed_qa: {len(pubmed)} rows")
print(f"  medmcqa:   {len(medmcqa)} rows")

Loading medalpaca...


README.md: 0.00B [00:00, ?B/s]

C:\Users\manis\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\manis\.cache\huggingface\hub\datasets--medalpaca--medical_meadow_medqa. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


medical_meadow_medqa.json:   0%|          | 0.00/10.7M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/10178 [00:00<?, ? examples/s]

...

data/validation-00000-of-00001.parquet:   0%|          | 0.00/1.48M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/182822 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/6150 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/4183 [00:00<?, ? examples/s]

✅ All datasets loaded!
  medalpaca: 10178 rows
  pubmed_qa: 1000 rows
  medmcqa:   182822 rows


In [8]:
qa_pairs = []

# --- medalpaca ---
for row in medalpaca:
    q = str(row.get("input", "") or "").strip()
    a = str(row.get("output", "") or "").strip()
    if q and a:
        qa_pairs.append(f"<question> {q} </question>\n<answer> {a} </answer>")

print(f"After medalpaca: {len(qa_pairs)} pairs")

# --- pubmed_qa ---
for row in pubmed:
    q = str(row.get("question", "") or "").strip()
    # long_answer is the full text answer
    a = str(row.get("long_answer", "") or "").strip()
    if not a:
        a = str(row.get("final_decision", "") or "").strip()
    if q and a:
        qa_pairs.append(f"<question> {q} </question>\n<answer> {a} </answer>")

print(f"After pubmed_qa: {len(qa_pairs)} pairs")

# --- medmcqa ---
options_map = {0: "A", 1: "B", 2: "C", 3: "D"}
for row in medmcqa:
    q = str(row.get("question", "") or "").strip()
    correct_idx = row.get("cop", None)  # correct option index (0-3)
    opa = str(row.get("opa", "") or "")
    opb = str(row.get("opb", "") or "")
    opc = str(row.get("opc", "") or "")
    opd = str(row.get("opd", "") or "")
    exp = str(row.get("exp", "") or "").strip()  # explanation

    options = [opa, opb, opc, opd]
    if q and correct_idx is not None and 0 <= correct_idx <= 3:
        correct_text = options[correct_idx]
        answer_letter = options_map[correct_idx]
        a = f"The correct answer is {answer_letter}: {correct_text}."
        if exp:
            a += f" {exp}"
        qa_pairs.append(f"<question> {q} </question>\n<answer> {a} </answer>")

print(f"After medmcqa: {len(qa_pairs)} pairs")
print(f"\n✅ Total Q&A pairs: {len(qa_pairs)}")

After medalpaca: 10178 pairs
After pubmed_qa: 11178 pairs
After medmcqa: 194000 pairs

✅ Total Q&A pairs: 194000


In [9]:
import random

random.seed(42)
random.shuffle(qa_pairs)

split_idx = int(len(qa_pairs) * 0.9)
train_data = qa_pairs[:split_idx]
val_data   = qa_pairs[split_idx:]

print(f"Train: {len(train_data)} pairs")
print(f"Val:   {len(val_data)} pairs")

Train: 174600 pairs
Val:   19400 pairs


In [10]:
with open("train.txt", "w", encoding="utf-8") as f:
    f.write("\n\n".join(train_data))

with open("val.txt", "w", encoding="utf-8") as f:
    f.write("\n\n".join(val_data))

print("✅ Saved!")
print(f"  train.txt → {len(train_data)} Q&A pairs")
print(f"  val.txt   → {len(val_data)} Q&A pairs")

# Quick preview
print("\n--- Sample from train.txt ---")
print(train_data[0])

✅ Saved!
  train.txt → 174600 Q&A pairs
  val.txt   → 19400 Q&A pairs

--- Sample from train.txt ---
<question> Extra capsular extraction of lens is not possible in: </question>
<answer> The correct answer is C: Lens subluxation. Ans: C (Lens subluxation) Ref: www.eiournalofophthalmology.com/ejo/ejo18.htmlv; BASAK Essentials of Ophthalmology,5th edition, pg no 254,260Explanation:ECCE - It involves the excision of the central part of the anterior capsule Followed by the expression of the nucleus & placement of the IOL in the remaining capsular bag.Capsule with zonules should be intact so the IOL placed will be in pupillary axis.ICCE - The whole crystalline lens including the capsule is removed.Subluxated lens:Partially displaced lens still remaining in the pupillary region.Causes:Familial ectopia lentisEctopia lentis et papillaeAniridiaMarfan's syndrome Homocystinuria Weill marchesani syndromeManagement of subluxated lens:Zonular dialysis < 90 degrees - Capsular tension ring with ECCE90